### <b>Michael Burgess, Brandon Lowery, Jason Martin</b>
### <b>COSC 526</b>
### <b>Dr. Linder</b>

#### <b>Our Data</b> (Airline Passenger Satisfaction): https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction

#### <b>Abstract</b>
This research investigates customer satisfaction in the realm of Airlines by analyzing various passenger demographic features, flight amenities, and more, predicting one "Satisfaction" variable. In our supervised approach, the study aims to: 
1) Use appropriate techniques to scale, normalize, clean, and engineer data
2) Apply machine learning techniques (baseline and advanced models) to evaluate our features against satisfaction
3) Evaluate the models, looking at performance and measuring precision, accuracy, AUC, etc; We include visualizations relating to our findings and the performance of our model
4) Expand on future research and potential opportunities to come of our findings, acknowledging the limitations and scope of what we researched
#### <b>Data Description / Dictionary</b> <i>taken from Kaggle</i>
- Gender: Gender of the passengers (Female, Male)
- Customer Type: The customer type (Loyal customer, disloyal customer)
- Age: The actual age of the passengers
- Type of Travel: Purpose of the flight of the passengers (Personal Travel, Business Travel)
- Class: Travel class in the plane of the passengers (Business, Eco, Eco Plus)
- Flight distance: The flight distance of this journey
- Inflight wifi service: Satisfaction level of the inflight wifi service (0:Not Applicable;1-5)
- Departure/Arrival time convenient: Satisfaction level of Departure/Arrival time convenient
- Ease of Online booking: Satisfaction level of online booking
- Gate location: Satisfaction level of Gate location
- Food and drink: Satisfaction level of Food and drink
- Online boarding: Satisfaction level of online boarding
- Seat comfort: Satisfaction level of Seat comfort
- Inflight entertainment: Satisfaction level of inflight entertainment
- On-board service: Satisfaction level of On-board service
- Leg room service: Satisfaction level of Leg room service
- Baggage handling: Satisfaction level of baggage handling
- Check-in service: Satisfaction level of Check-in service
- Inflight service: Satisfaction level of inflight service
- Cleanliness: Satisfaction level of Cleanliness
- Departure Delay in Minutes: Minutes delayed when departure
- Arrival Delay in Minutes: Minutes delayed when Arrival
- Satisfaction: Airline satisfaction level(Satisfaction, neutral or dissatisfaction)

#### <b>Data Prep and Feature Engineering</b>

In [23]:
# load data
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DataImport").getOrCreate()

df = spark.read.csv('data/train.csv', header=True, inferSchema=True, multiLine=True)
df = df.drop('_c0', 'id')


spark.sparkContext.setLogLevel("ERROR")

In [24]:
# EDA
from pyspark.sql import functions as F

df.printSchema()

print(df.dtypes)

print(df.columns)

df.describe().show()

df.summary().show()

df.show(5, truncate=True)

df.select([F.approx_count_distinct(c).alias(c) for c in df.columns]).show()

print('null check:')
df.select([F.count(F.when(df[c].isNull(), c)).alias(c) for c in df.columns]).show()

root
 |-- Gender: string (nullable = true)
 |-- Customer Type: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Type of Travel: string (nullable = true)
 |-- Class: string (nullable = true)
 |-- Flight Distance: integer (nullable = true)
 |-- Inflight wifi service: integer (nullable = true)
 |-- Departure/Arrival time convenient: integer (nullable = true)
 |-- Ease of Online booking: integer (nullable = true)
 |-- Gate location: integer (nullable = true)
 |-- Food and drink: integer (nullable = true)
 |-- Online boarding: integer (nullable = true)
 |-- Seat comfort: integer (nullable = true)
 |-- Inflight entertainment: integer (nullable = true)
 |-- On-board service: integer (nullable = true)
 |-- Leg room service: integer (nullable = true)
 |-- Baggage handling: integer (nullable = true)
 |-- Checkin service: integer (nullable = true)
 |-- Inflight service: integer (nullable = true)
 |-- Cleanliness: integer (nullable = true)
 |-- Departure Delay in Minutes: integer 

+-------+------+-----------------+------------------+---------------+--------+------------------+---------------------+---------------------------------+----------------------+------------------+------------------+-----------------+------------------+----------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------------+------------------------+--------------------+
|summary|Gender|    Customer Type|               Age| Type of Travel|   Class|   Flight Distance|Inflight wifi service|Departure/Arrival time convenient|Ease of Online booking|     Gate location|    Food and drink|  Online boarding|      Seat comfort|Inflight entertainment|  On-board service|  Leg room service|  Baggage handling|   Checkin service|  Inflight service|       Cleanliness|Departure Delay in Minutes|Arrival Delay in Minutes|        satisfaction|
+-------+------+-----------------+------------------+---------------+-------

+-------+------+-----------------+------------------+---------------+--------+------------------+---------------------+---------------------------------+----------------------+------------------+------------------+-----------------+------------------+----------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------------+------------------------+--------------------+
|summary|Gender|    Customer Type|               Age| Type of Travel|   Class|   Flight Distance|Inflight wifi service|Departure/Arrival time convenient|Ease of Online booking|     Gate location|    Food and drink|  Online boarding|      Seat comfort|Inflight entertainment|  On-board service|  Leg room service|  Baggage handling|   Checkin service|  Inflight service|       Cleanliness|Departure Delay in Minutes|Arrival Delay in Minutes|        satisfaction|
+-------+------+-----------------+------------------+---------------+-------

In [21]:
from pyspark.ml.feature import Imputer

# use median to fill missing data
imputer = Imputer(inputCols=['Arrival Delay in Minutes'], outputCols=['Arrival Delay in Minutes']).setStrategy('median')

In [26]:
from pyspark.ml.feature import StringIndexer

# encode binary features, only two options
gender_indexer = StringIndexer(inputCol='Gender', outputCol='gender_indexed', handleInvalid='keep')
customer_indexer = StringIndexer(inputCol='Customer Type', outputCol='customer_type_indexed', handleInvalid='keep')
travel_indexer = StringIndexer(inputCol='Type of Travel', outputCol='type_of_travel_indexed', handleInvalid='keep')

In [29]:
from pyspark.ml.feature import OneHotEncoder

# index/ohe class
class_indexer = StringIndexer(inputCol='Class', outputCol='class_indexed', handleInvalid='keep')
class_encoder = OneHotEncoder(inputCol='class_indexed', outputCol='class_encoded')

# index target var
satisfaction_indexer = StringIndexer(inputCol='satisfaction', outputCol='satisfaction_indexed')

In [28]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# scale continuous features
continuous_cols = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
cont_assembler = VectorAssembler(inputCols=continuous_cols, outputCol='cont_features_raw')
cont_scaler = StandardScaler(inputCol='cont_features_raw', outputCol='cont_features_scaled', withStd=True, withMean=True)

In [30]:
# survey features
final_assembler_inputs = [
    'cont_features_scaled',
    'Inflight wifi service', 
    'Departure/Arrival time convenient', 
    'Ease of Online booking', 
    'Gate location', 
    'Food and drink', 
    'Online boarding', 
    'Seat comfort', 
    'Inflight entertainment', 
    'On-board service', 
    'Leg room service', 
    'Baggage handling', 
    'Checkin service', 
    'Inflight service', 
    'Cleanliness',
    'gender_indexed',
    'customer_type_indexed',
    'type_of_travel_indexed',
    'class_encoded'
]

final_assembler = VectorAssembler(inputCols=final_assembler_inputs, outputCol='features')

In [32]:
from pyspark.ml import Pipeline

# build and run pipeline
stages = [
    imputer, 
    class_indexer, 
    class_encoder, 
    satisfaction_indexer,
    cont_assembler, 
    cont_scaler, 
    gender_indexer, 
    customer_indexer, 
    travel_indexer, 
    final_assembler
]

pipeline = Pipeline(stages=stages)
pipeline_model = pipeline.fit(df)
pipeline_df = pipeline_model.transform(df)

# filter down training data
final_df = pipeline_df.select('features', 'satisfaction_indexed')
final_df.show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+
|features                                                                                                                                                           |satisfaction_indexed|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+
|[-1.7452708978574245,-0.7315352402470835,0.2663913635665751,0.0741688031523776,3.0,4.0,3.0,1.0,5.0,3.0,5.0,5.0,4.0,3.0,4.0,4.0,5.0,5.0,1.0,0.0,1.0,0.0,0.0,1.0]    |0.0                 |
|[-0.9513556600584683,-0.9571789384183751,-0.36137307915636413,-0.23631164980054745,3.0,2.0,3.0,3.0,1.0,3.0,1.0,1.0,1.0,5.0,3.0,1.0,4.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0]|0.0                 |
|[-0.8851960569085553,-0.04758411956787989,-0.38752993093648663,-